In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import random
import torch.optim as optim

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def read_data(filename, columns):
    df = pd.read_csv("/kaggle/input/datasets/odedgolden/movielens-1m-dataset/"+filename,sep="::",engine="python",encoding="latin-1",header=None, names=columns)
    return df

In [ ]:
df_movies = read_data("movies.dat",columns=["ItemID", "Title","Genres"])
df_ratings = read_data("ratings.dat",columns=["UserID", "ItemID", "Rating","Timestamp"])
df_users = read_data("users.dat",columns=["UserID","Gender","Age","Occupation","ZipCode"])
df_users

In [ ]:
all_genres = set()
for g in df_movies['Genres']:
    all_genres.update(g.split('|'))
genre2id = {g:i for i,g in enumerate(sorted(all_genres))}
genre2id

In [ ]:
df_movies['GenreIDs'] = df_movies['Genres'].apply(lambda r: [genre2id[genre] for genre in r.split('|')])
df_movies

In [ ]:
df_users

In [ ]:
NUM_USERS = len(df_users['UserID'].unique())
NUM_ITEMS = len(df_movies['ItemID'].unique())

In [ ]:
def loo_split(ratings):
    """ Leave one out per user for testing """
    ratings = ratings.sort_values('Timestamp')
    test = ratings.groupby('UserID').last().reset_index()
    test_set = set(zip(test['UserID'],test['ItemID']))
    train = ratings[~ratings.apply(lambda r: (r.UserID, r.ItemID) in test_set,axis=1)]
    return train,test

In [ ]:
df_train, df_test = loo_split(df_ratings)

In [ ]:
# for negative sampling, track each user
user_history = defaultdict(set)

for _, r in df_train.iterrows():
    user_history[r.UserID].add(r.ItemID)

In [ ]:
class TwoTowerDataset(Dataset):
    def __init__(self, user, item, ratings,num_negatives=4):

        # ─── 1. Create mappings FIRST ───────────────────────────
        self.user2idx = {u: i for i, u in enumerate(user['UserID'].unique())}
        self.item2idx = {i: j for j, i in enumerate(item['ItemID'].unique())}

        # ─── 2. Map ratings data ────────────────────────────────
        self.user_ids = ratings['UserID'].map(self.user2idx).values
        self.item_ids = ratings['ItemID'].map(self.item2idx).values
        self.rating = ratings['Rating'].values

        # ─── 3. Age bucket mapping ──────────────────────────────
        def age_to_bucket(age):
            if age < 18: return 0
            elif age < 25: return 1
            elif age < 35: return 2
            elif age < 45: return 3
            elif age < 50: return 4
            elif age < 56: return 5
            else: return 6

        self.userid2age = {
            self.user2idx[row['UserID']]: age_to_bucket(row['Age'])
            for _, row in user.iterrows()
        }

        # ─── 4. Genre mapping (IMPORTANT: use mapped item_id) ───
        self.itemid2genre = {
            self.item2idx[row['ItemID']]: row['GenreIDs']
            for _, row in item.iterrows()
        }
        self.num_negatives = num_negatives
        self.all_item_indices = list(self.item2idx.values())

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        item_id = self.item_ids[idx]

        # Sample negatives not seen by this user
        seen = {self.item2idx[i] for i in user_history[user_id] 
                if i in self.item2idx}
        negs = []
        while len(negs) < self.num_negatives:
            cand = random.choice(self.all_item_indices)
            if cand not in seen:
                negs.append(cand)

        return {
            'UserID':      torch.tensor(user_id, dtype=torch.long),
            'User_Age':    torch.tensor(self.userid2age[user_id], dtype=torch.long),
            'ItemID':      torch.tensor(item_id, dtype=torch.long),
            'NegItemIDs':  torch.tensor(negs, dtype=torch.long),
            'GenreIDs':    torch.tensor(self.itemid2genre[item_id], dtype=torch.long),
        }
def collate_fn(batch):
    user_ids  = torch.stack([x['UserID']    for x in batch])
    user_ages = torch.stack([x['User_Age']  for x in batch])
    item_ids  = torch.stack([x['ItemID']    for x in batch])
    neg_ids   = torch.stack([x['NegItemIDs'] for x in batch])   # [B, num_neg]

    # ── Positive genres ──────────────────────────────────────
    genre_lists = [x['GenreIDs'] for x in batch]
    flat_genres = torch.cat(genre_lists)
    offsets = torch.tensor(
        [0] + [len(g) for g in genre_lists[:-1]], dtype=torch.long
    ).cumsum(0)

    return {
        'UserID':       user_ids,
        'User_Age':     user_ages,
        'ItemID':       item_ids,
        'NegItemIDs':   neg_ids,
        'GenreIDs':     flat_genres,
        'GenreOffsets': offsets,
    }

def bpr_loss(pos_scores, neg_scores):
    # pos_scores: [B], neg_scores: [B, num_neg]
    loss = -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-8)
    return loss.mean()

In [ ]:
dataset = TwoTowerDataset(df_users, df_movies, df_train)
loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=2,collate_fn = collate_fn)
print(f"Batches per epoch: {len(loader)}")


In [ ]:
class UserTower(nn.Module):
    def __init__(self, num_users,num_age_buckets=7, user_id_dim=32,age_dim =8, hidden_dim=[128,64], output_dim=64, dropout=0.2):
        super().__init__()

        # Input embedding
        self.user_id_embedding = nn.Embedding(num_users, user_id_dim)
        self.age_embedding = nn.Embedding(num_age_buckets, age_dim)

        # MLP Layers
        input_dim = user_id_dim + age_dim
        layers = []
        in_d = input_dim
        for out_d in hidden_dim:
            layers.extend([
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            in_d = out_d
        layers.append(nn.Linear(in_d, output_dim))
        self.mlp = nn.Sequential(*layers)

        nn.init.xavier_uniform_(self.user_id_embedding.weight)
        nn.init.xavier_uniform_(self.age_embedding.weight)

    def forward(self, user_ids, age_buckets):
        uid = self.user_id_embedding(user_ids)
        age = self.age_embedding(age_buckets)

        x = torch.cat([uid, age], dim=1)
        x = self.mlp(x)
        x = F.normalize(x, p=2, dim=1)
        return x
        

In [ ]:
class ItemTower(nn.Module):

    def __init__(
        self, 
        num_items,
        num_genres=18, 
        item_id_dim=32,
        genre_dim =8, 
        hidden_dim=[128,64], 
        output_dim=64, 
        dropout=0.2
    ):
        super().__init__()
        self.item_id_embedding = nn.Embedding(num_items, item_id_dim)
        self.genre_embedding = nn.EmbeddingBag(num_genres, genre_dim, mode='mean')

        # MLP Layers
        input_dim = item_id_dim + genre_dim
        in_d = input_dim
        layers=[]
        for out_d in hidden_dim:
            layers.extend([
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            in_d = out_d
        layers.append(nn.Linear(in_d, output_dim))
        self.mlp = nn.Sequential(*layers)

        nn.init.xavier_uniform_(self.item_id_embedding.weight)
        nn.init.xavier_uniform_(self.genre_embedding.weight)

    def forward(self, item_ids, genres, genre_offsets):
        iid = self.item_id_embedding(item_ids)
        genre = self.genre_embedding(genres, genre_offsets)
        x = torch.cat([iid,genre],dim=1)
        x = self.mlp(x)
        x = F.normalize(x,p=2,dim=1)
        return x

In [ ]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_age_buckets, num_items, num_genres, output_dim=64):
        super().__init__()
        self.user_tower = UserTower(num_users=num_users, num_age_buckets=num_age_buckets, output_dim=output_dim)
        self.item_tower = ItemTower(num_items=num_items, num_genres=num_genres, output_dim=output_dim)
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(0.07)))

    def forward(self, user_ids, num_age_buckets,item_ids, genres, genre_offsets):
        user_embs = self.user_tower(user_ids, num_age_buckets)
        item_embs = self.item_tower(item_ids, genres, genre_offsets)
        sim_matrix = torch.matmul(user_embs, item_embs.t())
        sim_matrix = sim_matrix * torch.exp(-self.log_temperature)

        return sim_matrix, user_embs, item_embs

    def get_user_embeddings(self, user_ids, num_age_buckets):
        self.eval() 
        with torch.no_grad():
            return self.user_tower(user_ids, num_age_buckets)
    def get_item_embeddings(self, item_ids, genres, genre_offsets):
        self.eval() 
        with torch.no_grad():
            return self.item_tower(item_ids, genres, genre_offsets)
            

In [ ]:
model = TwoTowerModel(num_users = NUM_USERS, num_items = NUM_ITEMS,num_age_buckets=7, num_genres=len(genre2id),output_dim=512).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
total_params

In [ ]:
def in_batch_negative_loss(similarity_matrix):
    batch_size = similarity_matrix.size(0)
    labels = torch.arange(batch_size, device=similarity_matrix.device)
    loss = F.cross_entropy(similarity_matrix, labels)
    return loss

In [ ]:
dummy_sim = torch.randn(8, 8) 
dummy_labels = torch.arange(8) 
init_loss = F.cross_entropy(dummy_sim, dummy_labels) 
print(f"Expected random init loss ≈ log(8) = {np.log(8):.3f}") 
print(f"Actual init loss: {init_loss.item():.3f}")

In [ ]:
item_genre_lookup = {}  # mapped_item_id → list of genre ids
for _, row in df_movies.iterrows():
    mapped = dataset.item2idx.get(row['ItemID'])
    if mapped is not None:
        item_genre_lookup[mapped] = row['GenreIDs']

In [ ]:
def get_neg_item_embeddings(model, neg_ids_flat, item_genre_lookup, device):
    """Pass negative items through the full ItemTower (not just embedding layer)."""
    genre_lists = [item_genre_lookup.get(idx.item(), [0]) for idx in neg_ids_flat]
    flat_genres = torch.tensor(
        [g for gl in genre_lists for g in gl], dtype=torch.long
    ).to(device)
    offsets = torch.tensor(
        [0] + [len(g) for g in genre_lists[:-1]], dtype=torch.long
    ).cumsum(0).to(device)
    return model.item_tower(neg_ids_flat, flat_genres, offsets) 
    
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0

    for batch in loader:
        user_ids      = batch['UserID'].to(device)
        age           = batch['User_Age'].to(device)
        item_ids      = batch['ItemID'].to(device)
        neg_ids       = batch['NegItemIDs'].to(device)       # [B, num_neg]
        genres        = batch['GenreIDs'].to(device)
        genre_offsets = batch['GenreOffsets'].to(device)

        optimizer.zero_grad()

        sim_matrix, user_embs, pos_item_embs = model(
            user_ids, age, item_ids, genres, genre_offsets
        )

        # ── In-batch contrastive loss ──────────────────────────────────
        inbatch_loss = in_batch_negative_loss(sim_matrix)

        # ── BPR loss with explicit negatives ──────────────────────────
        pos_scores = (user_embs * pos_item_embs).sum(dim=1)   # [B]

        B, num_neg = neg_ids.shape
        neg_ids_flat = neg_ids.view(-1)                        # [B*num_neg]

        # ✅ FIX: run through full item tower so dim matches user_embs (128)
        neg_item_embs = get_neg_item_embeddings(
            model, neg_ids_flat, item_genre_lookup, device
        ).view(B, num_neg, -1)                                 # [B, num_neg, D]

        neg_scores = torch.bmm(
            user_embs.unsqueeze(1),           # [B, 1, D]
            neg_item_embs.transpose(1, 2)     # [B, D, num_neg]
        ).squeeze(1)                           # [B, num_neg]

        bpr = bpr_loss(pos_scores, neg_scores)
        loss = inbatch_loss + 0.5 * bpr

        # ── Accuracy (diagnostic) ─────────────────────────────────────
        batch_size = sim_matrix.size(0)
        labels = torch.arange(batch_size, device=device)
        acc = (sim_matrix.argmax(dim=1) == labels).float().mean().item()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc  += acc

    n = len(loader)
    return total_loss / n, total_acc / n

In [ ]:
# ─── Run training ────────────────────────────────────────────────────────── 
N_EPOCHS  = 50                      # was 20
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# Linear warmup for 5 epochs, then cosine decay
def lr_lambda(epoch):
    if epoch < 5:
        return epoch / 5            # warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - 5) / (N_EPOCHS - 5)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


best_loss = float('inf') 
print("🚀 Training Two-Tower Model...") 
for epoch in range(1, N_EPOCHS + 1): 
    train_loss, train_acc = train_epoch(model, loader, optimizer, device) 
    scheduler.step() 
    if train_loss < best_loss: 
        best_loss = train_loss 
        torch.save(model.state_dict(), 'two_tower_best.pt')  
    print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | In-batch Acc: {train_acc:.3f}") 
print("✅ Training complete!")

In [ ]:
import numpy as np
import torch

def age_to_bucket(age):
    if age < 18: return 0
    elif age < 25: return 1
    elif age < 35: return 2
    elif age < 45: return 3
    elif age < 50: return 4
    elif age < 56: return 5
    else: return 6


def build_item_index(model, item_features, device, batch_size=512):
    """
    Builds item embedding matrix using ItemTower.
    item_features must contain:
        - ItemID (mapped index via dataset.item2idx)
        - GenreIDs (list[int])
    """

    # ── Pre-compute mapped IDs ONCE, outside the loop ──────────────────
    item_features = item_features.copy()
    item_features['MappedItemID'] = item_features['ItemID'].map(dataset.item2idx)

    model.eval()
    all_item_embs = []
    n_items = len(item_features)

    with torch.no_grad():
        for start in range(0, n_items, batch_size):
            end = min(start + batch_size, n_items)
            chunk = item_features.iloc[start:end]

            # ─── Item IDs  ← FIX: use chunk, not all of df_movies ──────
            item_ids = torch.tensor(
                chunk['MappedItemID'].values, dtype=torch.long
            ).to(device)

            # ─── Genres (flatten + offsets) ─────────────────────────────
            genre_lists = chunk['GenreIDs'].tolist()

            flat_genres = torch.tensor(
                [g for sublist in genre_lists for g in sublist],
                dtype=torch.long
            ).to(device)

            offsets = torch.tensor(
                [0] + [len(g) for g in genre_lists[:-1]],
                dtype=torch.long
            ).cumsum(0).to(device)

            # ─── Forward pass ────────────────────────────────────────────
            item_embs = model.get_item_embeddings(
                item_ids, flat_genres, offsets
            )  # [chunk_size, D]

            all_item_embs.append(item_embs.cpu().numpy())

    return np.vstack(all_item_embs)  # [num_items, D]

# Load best model 
model.load_state_dict(torch.load('two_tower_best.pt', map_location=device))
model.to(device)
item_embeddings = build_item_index(model, df_movies, device)
print(f"Item index shape: {item_embeddings.shape}")
 

def recommend_for_user(
    model,
    user_id,           # pass a valid ID, e.g. 1 (not 0)
    item_embeddings,
    user_features,
    age_to_bucket,     # this is a FUNCTION — call it with ()
    device,
    top_k=10,
    exclude_seen=None
):
    model.eval()

    with torch.no_grad():

        # ─── Get user row ───────────────────────────────────────────
        user_row = user_features[user_features['UserID'] == user_id]
        if user_row.empty:
            raise ValueError(f"UserID {user_id} not found. "
                             f"Valid range: {user_features['UserID'].min()}–"
                             f"{user_features['UserID'].max()}")
        user_row = user_row.iloc[0]

        user2idx = {u: i for i, u in enumerate(user_features['UserID'].unique())}
        mapped_uid = user2idx[user_id]
        uid_t = torch.tensor([mapped_uid], dtype=torch.long).to(device)

        # ─── FIX 2: call age_to_bucket() as a function, not dict ───
        age_bucket = age_to_bucket(user_row['Age'])
        age_t = torch.tensor([age_bucket], dtype=torch.long).to(device)

        # ─── User embedding ─────────────────────────────────────────
        user_emb = model.get_user_embeddings(uid_t, age_t).cpu().numpy()  # [1, D]

        # ─── Similarity (dot product = cosine for unit vectors) ─────
        scores = (user_emb @ item_embeddings.T).squeeze()  # [num_items]

        # ─── Exclude seen items ─────────────────────────────────────
        if exclude_seen is not None:
            for seen_item in exclude_seen:
                if seen_item < len(scores):
                    scores[seen_item] = -1e9

        # ─── Top-K ──────────────────────────────────────────────────
        top_k_items = np.argsort(scores)[::-1][:top_k]
        top_k_scores = scores[top_k_items]

    return top_k_items, top_k_scores
# MovieLens UserIDs start at 1 — use dataset.user2idx keys to pick a valid one
seen = user_history[1]   # ← was 0, now 1

top_items, top_scores = recommend_for_user(
    model,
    user_id=1,           # ← was 0
    item_embeddings=item_embeddings,
    user_features=df_users,
    age_to_bucket=age_to_bucket,
    device=device,
    top_k=10,
    exclude_seen=seen
)

print("\n🎯 Top-10 recommendations for user 0:")
for rank, (item, score) in enumerate(zip(top_items, top_scores), 1):
    print(f"Rank {rank}: item_id={item}, score={score:.4f}")

In [ ]:
pip install faiss-cpu

In [ ]:

import faiss 
def build_faiss_index(item_embeddings): 
    """ Build an ANN index over all item embeddings. 
    In production, this runs on hundreds of millions of items and is queried in ~1ms instead of O(N) brute-force. 
    IndexFlatIP = exact inner product (dot product) search For approximate (faster): IndexIVFFlat, IndexHNSW, IndexIVFPQ """ 
    dim = item_embeddings.shape[1] # 64 # Normalize to unit sphere (if not already done — we did it in the model) 
    embs_normed = item_embeddings / np.linalg.norm( item_embeddings, axis=1, keepdims=True ) # IndexFlatIP: exact inner product (= cosine sim for unit vectors) 
    index = faiss.IndexFlatIP(dim) 
    index.add(embs_normed.astype(np.float32)) 
    print(f"FAISS index built: {index.ntotal} items indexed") 
    return index 
    
def faiss_recommend(user_emb_np, index, top_k=10): 
    """ ANN search: find top-K items for a user embedding. 
    user_emb_np: numpy array [1, dim] 
    Returns: item indices and similarity scores """ 
# FAISS expects float32 
    query = user_emb_np.astype(np.float32) 
    D, I = index.search(query, top_k)

    return I[0], D[0]
    # D = distances (cosine similarities), I = item indices D, I = index.search(query, top_k) 
    # ~1ms for millions of items return I[0], D[0] 
    # Build and query 
faiss_index = build_faiss_index(item_embeddings)

# ─── Get user embedding for user_id=1 ──────────────────────────────
model.eval()
with torch.no_grad():
    user_id = 1                                         # valid MovieLens ID (1-indexed)
    user_row = df_users[df_users['UserID'] == user_id].iloc[0]

    user2idx = {u: i for i, u in enumerate(df_users['UserID'].unique())}
    mapped_uid = user2idx[user_id]

    uid_t = torch.tensor([mapped_uid], dtype=torch.long).to(device)
    age_t  = torch.tensor([age_to_bucket(user_row['Age'])], dtype=torch.long).to(device)

    u_emb = model.get_user_embeddings(uid_t, age_t).cpu().numpy()  # [1, D]

items, scores = faiss_recommend(u_emb, faiss_index, top_k=10)
print("FAISS top-10:", items)

In [ ]:
def evaluate_retrieval(model, test_df, item_embeddings, user_features, device, K=10):
    model.eval()
    user2idx = dataset.user2idx
    item2idx = dataset.item2idx

    hr_list, ndcg_list = [], []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            user_id  = row['UserID']
            pos_item = item2idx[row['ItemID']]

            user_row   = user_features[user_features['UserID'] == user_id].iloc[0]
            uid_t      = torch.tensor([user2idx[user_id]], dtype=torch.long).to(device)
            age_t      = torch.tensor([age_to_bucket(user_row['Age'])], dtype=torch.long).to(device)
            user_emb   = model.get_user_embeddings(uid_t, age_t).cpu().numpy()

            scores = (user_emb @ item_embeddings.T).squeeze()

            # ── Mask seen items ──────────────────────────────────
            seen = {item2idx[i] for i in user_history[user_id] if i in item2idx}
            for s in seen:
                scores[s] = -1e9

            top_k = np.argsort(scores)[::-1][:K].tolist()

            hit = 1 if pos_item in top_k else 0
            hr_list.append(hit)
            ndcg_list.append(1.0 / np.log2(top_k.index(pos_item) + 2) if hit else 0.0)

    return {f'HR@{K}': np.mean(hr_list), f'NDCG@{K}': np.mean(ndcg_list)}
metrics = evaluate_retrieval(model, df_test, item_embeddings, df_users, device, K=10) 
print("\n📊 Two-Tower Evaluation (full catalog):") 
for k, v in metrics.items(): 
    print(f" {k}: {v:.4f}")